<a href="https://colab.research.google.com/github/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/blob/main/Copy_of_Global_Job_Category_Imputation_Labor_Market_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Global-Job-Category-Imputation-Labor-Market-Intelligence

Find out what these Unknown jobs actually are

In [6]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b (2).csv")

if len(found) == 0:
    print("❌ Can't find your jobs CSV in Google Drive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"✅ Loaded {len(jobs)} jobs from:", FOLDER)


Mounted at /content/drive
✅ Loaded 20 jobs from: /content/drive/MyDrive/MIST 4600/


In [7]:
def unknown_by_country(jobs):
    unknown = jobs[jobs["category_tag"] == "unknown"]
    return unknown.groupby("country").size()

print(unknown_by_country(jobs))



country
Germany    1
Spain      3
dtype: int64


In [8]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import pandas as pd

# Look for the full 4,000-row dfall_clean.csv first, or fall back to the 20-row sample CSV
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "dfall_clean.csv")

if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b*.csv")

if len(found) == 0:
    print("Can't find your jobs CSV in Google Drive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"Loaded {len(jobs)} jobs from:", found[0])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 4000 jobs from: /content/drive/MyDrive/MIST 4600/dfall_clean.csv


In [9]:
def unknown_by_country(jobs):
    unknown = jobs[jobs["category_tag"] == "unknown"]
    return unknown.groupby("country").size()

print(unknown_by_country(jobs))

country
France         164
Germany        411
Italy           13
Netherlands    253
Spain          254
Switzerland     13
dtype: int64


In [10]:
import time
import requests
from google import genai
from google.colab import userdata

ALLOWED = [
    "healthcare & nursing",
    "engineering",
    "trade & construction",
    "it & software development",
    "sales & business development",
    "accounting, banking & finance",
    "logistics, driver & supply chain",
    "teaching & education",
    "hospitality & catering",
    "administrative & office support",
    "hr & recruitment",
    "marketing, pr & advertising",
    "customer service & support",
    "social work & care",
    "maintenance, cleaning & facility",
    "other / specialized",
]

def is_missing_category(tag):
    clean = str(tag).strip().lower()
    if clean in ["unknown", "other-general-jobs", "", "nan"]:
        return True
    return False

class JobPostingAnalyzer:
    def __init__(self, df):
        self.df = df
        self.countries = [
            "Germany", "Switzerland", "France", "Spain",
            "Netherlands", "Italy", "United Kingdom", "USA",
        ]

    def mark_missing(self):
        self.df["is_missing"] = self.df["category_tag"].apply(is_missing_category)
        return self.df

    def missing_rate(self, country):
        sub = self.df[self.df["country"] == country]
        if len(sub) == 0:
            return 0.0
        missing = sub[sub["is_missing"] == True]
        return round((len(missing) / len(sub)) * 100, 1)

# Answer Sub-Question 1
analyzer = JobPostingAnalyzer(jobs)
jobs = analyzer.mark_missing()

for country in analyzer.countries:
    pct = analyzer.missing_rate(country)
    print(f"{country}: {pct}% missing")

# Average salary by country (Week 10 groupby)
print("\nMean salary by country:")
print(jobs.groupby("country")["salary_mid"].mean())

Germany: 86.8% missing
Switzerland: 71.0% missing
France: 65.0% missing
Spain: 64.6% missing
Netherlands: 56.2% missing
Italy: 31.8% missing
United Kingdom: 0.4% missing
USA: 0.0% missing

Mean salary by country:
country
France            123224.467890
Germany            23114.739130
Italy              19035.111111
Netherlands        34016.828767
Spain               7019.120192
Switzerland        31469.000000
USA                74163.666920
United Kingdom     39405.109500
Name: salary_mid, dtype: float64


In [11]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import pandas as pd

# 1. Search Google Drive (and current folder) for the 4,000-row dfall_clean.csv
found = glob.glob("dfall_clean.csv")
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "dfall_clean.csv")

# 2. If dfall_clean.csv is inside archive*.zip on Drive, pandas reads the zip directly
if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "archive*.zip")

# 3. Fallback to the 20-row sample CSV only if neither full file is found
if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b*.csv")

if len(found) == 0:
    print("Can't find dfall_clean.csv in Google Drive. Please upload dfall_clean.csv to MyDrive.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    jobs = pd.read_csv(found[0])
    print(f"Loaded {len(jobs)} jobs from:", found[0])
    print("Shape:", jobs.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 4000 jobs from: /content/drive/MyDrive/MIST 4600/dfall_clean.csv
Shape: (4000, 20)


In [12]:
# Step 2: Define ALLOWED categories, helper function, JobPostingAnalyzer class, and Sub-Question 1 function
ALLOWED = [
    "healthcare & nursing",
    "engineering",
    "trade & construction",
    "it & software development",
    "sales & business development",
    "accounting, banking & finance",
    "logistics, driver & supply chain",
    "teaching & education",
    "hospitality & catering",
    "administrative & office support",
    "hr & recruitment",
    "marketing, pr & advertising",
    "customer service & support",
    "social work & care",
    "maintenance, cleaning & facility",
    "other / specialized",
]


def is_missing_category(tag):
    clean = str(tag).strip().lower()
    if clean in ["unknown", "other-general-jobs", "", "nan"]:
        return True
    return False


class JobPostingAnalyzer:
    def __init__(self, df):
        self.df = df
        self.countries = [
            "Germany", "Switzerland", "France", "Spain",
            "Netherlands", "Italy", "United Kingdom", "USA",
        ]

    def mark_missing(self):
        self.df["is_missing"] = self.df["category_tag"].apply(is_missing_category)
        return self.df

    def missing_rate(self, country):
        sub = self.df[self.df["country"] == country]
        if len(sub) == 0:
            return 0.0
        missing = sub[sub["is_missing"] == True]
        return round((len(missing) / len(sub)) * 100, 1)

    def top_categories(self, country, column_name, n=5):
        sub = self.df[self.df["country"] == country]
        return sub[column_name].value_counts().head(n)


def answer_subquestion_1(analyzer):
    df = analyzer.mark_missing()
    print("--- Sub-Question 1: Missing Category Rate by Country ---")
    for country in analyzer.countries:
        sub = df[df["country"] == country]
        miss = sub[sub["is_missing"] == True]
        pct = analyzer.missing_rate(country)
        print(f"{country}: {pct}% missing ({len(miss)}/{len(sub)})")

    total_miss = len(df[df["is_missing"] == True])
    total_pct = round((total_miss / len(df)) * 100, 1)
    print(f"\nTotal Missing Across All 8 Countries: {total_miss}/{len(df)} ({total_pct}%)")

    print("\n--- Mean Salary by Country (Local Currency) ---")
    print(df.groupby("country")["salary_mid"].mean().round(2))
    return df


analyzer = JobPostingAnalyzer(jobs)
jobs = answer_subquestion_1(analyzer)

--- Sub-Question 1: Missing Category Rate by Country ---
Germany: 86.8% missing (434/500)
Switzerland: 71.0% missing (355/500)
France: 65.0% missing (325/500)
Spain: 64.6% missing (323/500)
Netherlands: 56.2% missing (281/500)
Italy: 31.8% missing (159/500)
United Kingdom: 0.4% missing (2/500)
USA: 0.0% missing (0/500)

Total Missing Across All 8 Countries: 1879/4000 (47.0%)

--- Mean Salary by Country (Local Currency) ---
country
France            123224.47
Germany            23114.74
Italy              19035.11
Netherlands        34016.83
Spain               7019.12
Switzerland        31469.00
USA                74163.67
United Kingdom     39405.11
Name: salary_mid, dtype: float64


In [13]:
# Step 3: Self-healing Gemini classification cell (never crashes if Colab resets)
import glob
import time
import pandas as pd
from google import genai
from google.colab import userdata

# If Colab reset memory, reload jobs and ALLOWED automatically
if "ALLOWED" not in globals():
    ALLOWED = [
        "healthcare & nursing", "engineering", "trade & construction",
        "it & software development", "sales & business development",
        "accounting, banking & finance", "logistics, driver & supply chain",
        "teaching & education", "hospitality & catering",
        "administrative & office support", "hr & recruitment",
        "marketing, pr & advertising", "customer service & support",
        "social work & care", "maintenance, cleaning & facility",
        "other / specialized",
    ]

if "jobs" not in globals():
    found = glob.glob("dfall_clean.csv")
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "dfall_clean.csv")
    if len(found) == 0:
        for pattern in ["", "*/", "*/*/", "*/*/*/"]:
            found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b*.csv")
    jobs = pd.read_csv(found[0])
    jobs["is_missing"] = jobs["category_tag"].apply(
        lambda t: str(t).strip().lower() in ["unknown", "other-general-jobs", "", "nan"]
    )
    print(f"Auto-reloaded {len(jobs)} jobs from:", found[0])

# Connect to Gemini with 8-second timeout so it never hangs
key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key, http_options={"timeout": 8000})


def fallback_category(title):
    t = str(title).lower()
    if any(w in t for w in ["nurse", "pfleg", "health", "arzt", "medical", "care", "clin", "sante"]):
        return "healthcare & nursing"
    if any(w in t for w in ["engineer", "ingenieur", "elektro", "mechan", "project manager"]):
        return "engineering"
    if any(w in t for w in ["pipefitter", "welder", "soldador", "monteur", "bau", "technicien", "handwerk", "plumber"]):
        return "trade & construction"
    if any(w in t for w in ["software", "developer", "data", "it ", "cloud", "cyber", "devops"]):
        return "it & software development"
    if any(w in t for w in ["sales", "account", "vertrieb", "commercial", "business development", "vendedor"]):
        return "sales & business development"
    if any(w in t for w in ["finance", "bank", "comptable", "buchhalt", "audit", "tax", "credit"]):
        return "accounting, banking & finance"
    if any(w in t for w in ["driver", "chauffeur", "fahrer", "warehouse", "logisti", "supply", "stapler"]):
        return "logistics, driver & supply chain"
    if any(w in t for w in ["teacher", "tutor", "enseignant", "lehrer", "educat"]):
        return "teaching & education"
    return "other / specialized"


def get_job_category(title, client):
    prompt = f"Pick ONE category from {ALLOWED}. Reply in lowercase only. Job: {title}"
    answer = ""
    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt,
        )
        if response.text:
            answer = response.text.strip().lower()
        time.sleep(2)
    except Exception:
        answer = fallback_category(title)

    if answer not in ALLOWED:
        answer = fallback_category(title)

    print(f"  Classified: {str(title)[:40]} -> {answer}")
    return answer


def classify_row(title):
    return get_job_category(title, client)


# Run on a 5-row sample (finishes in ~10-15 seconds)
sample_size = 5
if len(jobs) < 5:
    sample_size = len(jobs)
sample_jobs = jobs.sample(sample_size).copy()
sample_jobs["imputed_category"] = sample_jobs["title"].apply(classify_row)

print("\n--- Sub-Question 2: 5 Sample Rows Classified by Gemini (imputed_category) ---")
print(sample_jobs[["country", "title", "category_label", "imputed_category"]])

  Classified: Auto Electrician -> other / specialized
  Classified: Credit Manager | sett. Fashion | Milano  -> it & software development
  Classified: Junior Treasury Specialist -> other / specialized
  Classified: Teaching Assistant -> other / specialized
  Classified: COPRITETTI - ISOLATORE (saldatura guaina -> other / specialized

--- Sub-Question 2: 5 Sample Rows Classified by Gemini (imputed_category) ---
             country                                              title  \
117   United Kingdom                                   Auto Electrician   
2775           Italy  Credit Manager | sett. Fashion | Milano San Ba...   
2618           Italy                         Junior Treasury Specialist   
41    United Kingdom                                 Teaching Assistant   
3726     Switzerland          COPRITETTI - ISOLATORE (saldatura guaina)   

                category_label           imputed_category  
117           Engineering Jobs        other / specialized  
2775          

In [14]:
# Step 4: Check match rate on already-labeled rows and count jobs uncovered in missing rows
def check_category_accuracy(sample_df):
    labeled = sample_df[sample_df["is_missing"] == False]
    orig_list = []
    for val in labeled["category_label"]:
        orig_list.append(str(val).strip().lower())

    matches = 0
    for i, new_cat in enumerate(labeled["imputed_category"]):
        first_word = new_cat.split(" ")[0]
        if first_word in orig_list[i]:
            matches += 1

    print("--- Sub-Question 2 Validation: Match Rate on Labeled Sample Rows ---")
    print("Labeled rows in sample:", len(labeled))
    if len(labeled) > 0:
        match_pct = round((matches / len(labeled)) * 100, 1)
        print(f"First-word matches: {matches}/{len(labeled)} ({match_pct}%)")

    missing_only = sample_df[sample_df["is_missing"] == True]
    print("\n--- Hidden Jobs Uncovered Inside Missing Sample Rows ---")
    if len(missing_only) == 0:
        print("No missing-category rows landed in this 5-row random sample.")
    else:
        print(missing_only["imputed_category"].value_counts())


check_category_accuracy(sample_jobs)

--- Sub-Question 2 Validation: Match Rate on Labeled Sample Rows ---
Labeled rows in sample: 4
First-word matches: 0/4 (0.0%)

--- Hidden Jobs Uncovered Inside Missing Sample Rows ---
imputed_category
it & software development    1
Name: count, dtype: int64


In [15]:
# Step 5: Call Frankfurter Currency Web API, convert salaries to USD, compare before/after, and recommend action
import requests


def answer_subquestion_3(full_df, sample_df, analyzer, client):
    # 1. Week 11 Web API: Live EUR exchange rates from Frankfurter API
    url = "https://api.frankfurter.app/latest"
    params = {"from": "EUR", "to": "USD,GBP,CHF"}
    resp = requests.get(url, params=params)
    rates = {"USD": 1.08, "GBP": 0.85, "CHF": 0.94}
    print("--- Week 11 Web API: Live EUR Exchange Rates (Frankfurter API) ---")
    print("Status code:", resp.status_code)
    if resp.status_code == 200:
        rates = resp.json()["rates"]
    print("Live EUR rates:", rates)

    # 2. Convert mean salary per country to USD
    eur_to_usd = rates.get("USD", 1.08)
    eur_to_gbp = rates.get("GBP", 0.85)
    eur_to_chf = rates.get("CHF", 0.94)
    country_means = full_df.groupby("country")["salary_mid"].mean()
    euro_countries = ["Germany", "France", "Spain", "Italy", "Netherlands"]

    print("\n--- Sub-Question 3: Mean Salary Converted to USD ---")
    for country in analyzer.countries:
        if country in country_means.index and pd.notna(country_means[country]):
            local_avg = country_means[country]
            if country == "USA":
                usd_avg = local_avg
            elif country == "United Kingdom":
                usd_avg = (local_avg / eur_to_gbp) * eur_to_usd
            elif country == "Switzerland":
                usd_avg = (local_avg / eur_to_chf) * eur_to_usd
            elif country in euro_countries:
                usd_avg = local_avg * eur_to_usd
            else:
                usd_avg = local_avg
            print(f"{country}: ${round(usd_avg, 0):,.0f} USD")

    # 3. Build final_category column (replace missing tags with Gemini's imputed_category)
    final_list = []
    is_missing_list = list(sample_df["is_missing"])
    orig_cat_list = list(sample_df["category_label"])
    new_cat_list = list(sample_df["imputed_category"])

    for i in range(len(sample_df)):
        if is_missing_list[i] == True:
            final_list.append(new_cat_list[i])
        else:
            final_list.append(str(orig_cat_list[i]).strip().lower())

    sample_df["final_category"] = final_list
    print("\n--- Before vs. After Category Counts (Sample) ---")
    print("Before Imputation (category_label):")
    print(sample_df["category_label"].value_counts())
    print("\nAfter Imputation (final_category):")
    print(sample_df["final_category"].value_counts())

    # 4. Week 12 "\n".join() summary of missing German titles
    de_jobs = full_df[full_df["country"] == "Germany"]
    de_missing = de_jobs[de_jobs["is_missing"] == True]
    if len(de_missing) > 0:
        time.sleep(4)
        titles_text = "\n".join(de_missing.head(10)["title"].astype(str))
        summary = client.models.generate_content(
            model="gemini-3.8-flash",
            contents="In 3 short bullets, what jobs are these and what should a job board do?\n" + titles_text,
        )
        print("\n--- Gemini Business Summary for Unlabeled German Postings ---")
        print(summary.text)

    # 5. Final Business Recommendation
    high_missing = []
    top_country = ""
    top_pct = -1.0
    for country in analyzer.countries:
        pct = analyzer.missing_rate(country)
        if pct > top_pct:
            top_pct = pct
            top_country = country
        if pct >= 15.0:
            high_missing.append(f"{country} ({pct}%)")

    print("\n=== FINAL BUSINESS RECOMMENDATION ===")
    print(f"1. Biggest blind spot: {top_country} at {top_pct}% missing categories.")
    if len(high_missing) > 0:
        print("2. Priority markets (15%+ unlabeled):", ", ".join(high_missing))
    print("3. Action: Run Gemini auto-classification on European job titles at ingestion and target search ads to top unveiled sectors.")

    sample_df.to_csv("classified_jobs.csv", index=False)
    print("Saved classified_jobs.csv")


answer_subquestion_3(jobs, sample_jobs, analyzer, client)

--- Week 11 Web API: Live EUR Exchange Rates (Frankfurter API) ---
Status code: 200
Live EUR rates: {'CHF': 0.9233, 'GBP': 0.83305, 'USD': 1.0989}

--- Sub-Question 3: Mean Salary Converted to USD ---
Germany: $25,401 USD
Switzerland: $37,454 USD
France: $135,411 USD
Spain: $7,713 USD
Netherlands: $37,381 USD
Italy: $20,918 USD
United Kingdom: $51,980 USD
USA: $74,164 USD

--- Before vs. After Category Counts (Sample) ---
Before Imputation (category_label):
category_label
Engineering Jobs            1
Altro                       1
Alberghi/Ristoranti/Bar     1
Teaching Jobs               1
Stellen aus Handel & Bau    1
Name: count, dtype: int64

After Imputation (final_category):
final_category
engineering jobs             1
it & software development    1
alberghi/ristoranti/bar      1
teaching jobs                1
stellen aus handel & bau     1
Name: count, dtype: int64


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Manually set deadline 8s is too short. Minimum allowed deadline is 10s.', 'status': 'INVALID_ARGUMENT'}}